# 2 · Query the Vector Search Index

Runs a similarity search against the index created in notebook `1` to confirm
retrieval works before wiring it into the Supervisor Agent.

### Define Parameters
Review the widgets before running. Defaults match the **Marketing Campaign Analytics** layout.
| Widget | Default | What it controls |
| --- | --- | --- |
| `CATALOG_NAME` | `marketing_campaign_analytics` | Unity Catalog **catalog** for all demo objects (tables, volumes, indexes). |
| `SCHEMA_NAME` | `marketing_campaign` | Schema inside the catalog for sales data and contracts. |
| `INDEX_NAME` | `contract_chunks_index` | Vector Search index to query (from notebook `1`). |
| `QUESTION` | *(see default)* | Similarity-search query text. |
| `NUM_RESULTS` | `5` | Number of chunks to return. |


In [ ]:
dbutils.widgets.text("CATALOG_NAME", "marketing_campaign_analytics", "CATALOG_NAME — Unity Catalog catalog for demo objects")
dbutils.widgets.text("SCHEMA_NAME", "marketing_campaign", "SCHEMA_NAME — schema inside the catalog")
dbutils.widgets.text("INDEX_NAME", "contract_chunks_index", "INDEX_NAME — Vector Search index (from notebook 1)")
dbutils.widgets.text("QUESTION", "What are the payment terms in the fictional_marketing_services_echo contract?", "QUESTION — similarity-search query text")
dbutils.widgets.text("NUM_RESULTS", "5", "NUM_RESULTS — chunks to return from similarity search")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
INDEX_NAME = dbutils.widgets.get("INDEX_NAME")
QUESTION = dbutils.widgets.get("QUESTION")
NUM_RESULTS = int(dbutils.widgets.get("NUM_RESULTS"))
INDEX_FQN = f"{CATALOG_NAME}.{SCHEMA_NAME}.{INDEX_NAME}"

### Similarity Search

In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

resp = w.vector_search_indexes.query_index(
    index_name=INDEX_FQN,
    columns=['file_name', 'chunk_index', 'content'],
    query_text=QUESTION,
    num_results=NUM_RESULTS,
)

data = (resp.result.data_array if resp.result else []) or []
print(f'Top {len(data)} matches for: {QUESTION}\n')
for row in data:
    fname, cidx, content, score = row[0], row[1], row[2], row[-1]
    print(f'[{float(score):.3f}] {fname} (chunk {cidx})')
    print('   ', str(content)[:200].replace(chr(10), ' '), '...')
    print()